Resources for analysis:

* https://ucdavis-bioinformatics-training.github.io/2022-June-RNA-Seq-Analysis/data_analysis/DE_Analysis_mm_with_quizzes_fixed
* https://ucdavis-bioinformatics-training.github.io/2022-June-RNA-Seq-Analysis/data_reduction/03-counts_mm
* https://github.com/ben-laufer/RNA-seq/blob/main/04-limma-voom.R
* https://raw.githubusercontent.com/ucdavis-bioinformatics-training/2020-mRNA_Seq_Workshop/master/data_analysis/enrichment_mm.Rmd
* https://bioconductor.org/packages/release/bioc/vignettes/biomaRt/inst/doc/accessing_ensembl.html#selecting-an-ensembl-biomart-database-and-dataset
* https://cran.r-project.org/web/packages/enrichR/vignettes/enrichR.html

# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(openxlsx)
library(readxl)
library(dplyr)
library(glue)
library(ggplot2)
library(ggrepel)

# Load Data

In [4]:
# List all DEG files in the subdirectories 
files <- list.files(path = "05_DEGs", pattern = "_results.csv$", full.names = TRUE, recursive = TRUE)

# Initialize an empty vector to hold files with significant DEGs
valid_files <- c()

# Loop through each file
for (file in files) {
  
  # Read the current file
  df <- read.csv(file, row.names = 1)
  
  # Extract comparison name 
  comparison <- gsub("_results.csv", "", basename(file))
  
  # Filter to only significant DEGs (adj.P.Val < 0.01)
  significant_df <- df[df$adj.P.Val < 0.01 & !is.na(df$adj.P.Val), ]
  
  # Check the number of significant DEGs
  num_significant <- nrow(significant_df)
  
  # If there are significant DEGs, keep the file
  if (num_significant > 0) {
    valid_files <- c(valid_files, file)
  }
}

# Update the original files list to only include valid files
files <- valid_files

# Volcano Plots

In [17]:
create_volcano_plot <- function(file) {
  
  # Extract sample type and comparison from file path
  comparison <- gsub("_results.csv$", "", basename(file))
  filename <- comparison
  print(filename)
  
  # Read the CSV file
  data <- read.csv(file, row.names = 1)
  
  # Check if there are any significant genes
  if (nrow(data) == 0) {
    cat("Empty data file:", file, "\n")
    return()
  }
  
  # Define significance thresholds
  adj_p_cutoff <- 0.01
  logfc_cutoff <- 1
  
  # Create consistent significance definition
  data$Significant <- ifelse(data$adj.P.Val < adj_p_cutoff & abs(data$logFC) > logfc_cutoff, "Yes", "No")
  
  # Create color categories
  data$Color <- ifelse(data$adj.P.Val < adj_p_cutoff & data$logFC > logfc_cutoff, "Upregulated", 
                       ifelse(data$adj.P.Val < adj_p_cutoff & data$logFC < -logfc_cutoff, "Downregulated", "Not Significant"))
  
  # Count significant genes
  n_up <- sum(data$Color == "Upregulated", na.rm = TRUE)
  n_down <- sum(data$Color == "Downregulated", na.rm = TRUE)
  
  cat("Upregulated:", n_up, "| Downregulated:", n_down, "\n")
  
  # Skip plotting if too few significant genes
  if (n_up < 3 && n_down < 3) {
    cat("Not enough significant DEGs for", filename, "- skipping\n")
    return()
  }
  
  # Get top 10 upregulated genes (by logFC)
  top_upregulated <- data %>%
    filter(Color == "Upregulated") %>%
    arrange(desc(logFC)) %>%
    head(10)
  
  # Get top 10 downregulated genes (by logFC)
  top_downregulated <- data %>%
    filter(Color == "Downregulated") %>%
    arrange(logFC) %>%
    head(10)
  
  # Combine top genes for labeling
  top_genes <- rbind(top_upregulated, top_downregulated)
  
  # Create output directory if it doesn't exist
  dir.create("00_FIGURES/Volcano_Plots/", showWarnings = FALSE, recursive = TRUE)
  
  # Initialize PDF
  pdf(file = paste0("00_FIGURES/Volcano_Plots/volcano_plot_", filename, ".pdf"), width = 10, height = 7)
  
  # Factor the Color column in correct order
  data$Color <- factor(data$Color, levels = c("Downregulated", "Not Significant", "Upregulated"))
  
  # Calculate axis limits
  x_min <- min(data$logFC, na.rm = TRUE) - 0.5
  x_max <- max(data$logFC, na.rm = TRUE) + 0.5
  y_max <- max(-log10(data$adj.P.Val), na.rm = TRUE) + 1

  # Format subtitle
  subtitle_text <- paste0(
    "(adj.P.Val < ", adj_p_cutoff,
    ", |logFC| > ", logfc_cutoff,
    ")\n",
    "Up: ", n_up,
    " | Down: ", n_down
  ) 
    
  # Generate volcano plot
  p <- ggplot(data, aes(x = logFC, y = -log10(adj.P.Val), color = Color)) +
    geom_point(alpha = 0.6, size = 1.5) +
    scale_color_manual(
      values = c("Downregulated" = "blue", "Not Significant" = "grey", "Upregulated" = "red"),
      name = "Significance"
    ) +
    labs(title = gsub("_", " ", comparison),
         subtitle = subtitle_text,
         x = "Log2 Fold Change",
         y = "-Log10 Adjusted P-Value") +
    scale_x_continuous(limits = c(x_min, x_max), expand = expansion(mult = c(0.05, 0.05))) +
    scale_y_continuous(expand = expansion(mult = c(0.05, 0.05))) +
    theme_minimal() +
    theme(legend.position = "top",
          plot.title = element_text(hjust = 0.5, size = 14),
          plot.subtitle = element_text(hjust = 0.5, size = 10)) +
    geom_hline(yintercept = -log10(adj_p_cutoff), linetype = "dashed", color = "darkgrey") +
    geom_vline(xintercept = c(-logfc_cutoff, logfc_cutoff), linetype = "dashed", color = "darkgrey")
  
  # Add labels if there are top genes
  if (nrow(top_genes) > 0) {
    p <- p + geom_label_repel(data = top_genes, aes(label = rownames(top_genes)),
                              size = 3,
                              box.padding = 0.5,
                              point.padding = 0.5,
                              segment.color = 'grey50',
                              show.legend = FALSE,
                              max.overlaps = 20)
  }
  
  # Print the plot
  print(p)
  
  # Close the PDF device
  dev.off()
}

In [18]:
# Create volcano plots
for (file in files) {
  create_volcano_plot(file)
}

[1] "MUT_Guide6_vs_MUT_NonEffector"
Upregulated: 715 | Downregulated: 1122 
[1] "MUT_Guide6_vs_MUT_NT"
Upregulated: 570 | Downregulated: 1751 
[1] "MUT_Guide6_vs_WT_NT"
Upregulated: 1083 | Downregulated: 2241 
[1] "MUT_NonEffector_vs_MUT_NT"
Upregulated: 707 | Downregulated: 1325 
[1] "MUT_NonEffector_vs_WT_NT"
Upregulated: 1041 | Downregulated: 1892 
[1] "MUT_NT_vs_WT_NT"
Upregulated: 795 | Downregulated: 943 
